# Decision Tree — Dengue vs Malaria (Leakage-Safe, Full-Test-Set Version)

**Corrections applied vs. `DT_modified_before_resampling.ipynb`:**
1. `WARD_encoded` is no longer read as a pre-baked leaked feature. It is treated as a raw
   ward identifier (`WARD_ID`) and re-encoded **inside** the pipeline, fit only on training
   folds (`WardTargetEncoder` below), with smoothing + unseen-category fallback.
2. All evaluation in the threshold loop now uses the **full test set**, not `.iloc[:10]`
   or `.iloc[:50]`.
3. Added: full confusion matrix, per-class precision/recall/F1, ROC-AUC, PR-AUC.
4. Added: optional 5-fold CV block at the end to report mean ± std for headline numbers.

**To adapt to your other 6 models:** replace the `DecisionTreeClassifier` + `param_grid` in
the "MODEL DEFINITION" cell with the classifier/grid from your existing RF / XGBoost / KNN /
LR / MLP / GaussianNB notebooks. Everything else (encoder, pipeline, evaluation, threshold
loop) is model-agnostic and does not need to change.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_validate
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score, f1_score,
    roc_auc_score, average_precision_score
)


## 1. Load data

Update the path to wherever you keep the file in your Drive/Colab session.

In [ ]:
# NOTE: use the RAW pre-encoded ward column if you have it (e.g. a WARD name/ID column
# from before target-encoding was ever applied). If you only have this file (where
# WARD_encoded already exists as a fixed numeric value per ward), that's fine too --
# there are exactly 144 unique values, one per KMC ward, so we can safely treat it as a
# categorical ward IDENTIFIER and recompute a fresh, leakage-safe encoding from it below.

df = pd.read_csv('/content/drive/MyDrive/Dengee_Maleria_Modified_XAI/encoded_dataset_28_5_25 (1).csv')

df = df.rename(columns={'WARD_encoded': 'WARD_ID'})
print(df['WARD_ID'].nunique(), 'unique wards')
df.head()


In [ ]:
X = df.drop('CLASS', axis=1)
y = df['CLASS']

numeric_cols = ['AGE', 'MONTH_sin', 'MONTH_cos']   # scaled directly
ward_col     = ['WARD_ID']                          # re-encoded via WardTargetEncoder below
passthrough_cols = [c for c in X.columns if c not in numeric_cols + ward_col]

print('Numeric (scaled):', numeric_cols)
print('Ward col (re-encoded):', ward_col)
print('Passthrough (binary/one-hot):', passthrough_cols)


## 2. Leakage-safe ward target encoder

Fits **only** on the data passed to `.fit(X, y)`. When used inside a `Pipeline` + `GridSearchCV`/`cross_validate`, sklearn automatically re-fits this on every training fold and only *transforms* (never fits on) the held-out fold -- so no test-set label information ever leaks into the encoding.

In [ ]:
class WardTargetEncoder(BaseEstimator, TransformerMixin):
    """
    Leakage-safe target encoder for a high-cardinality categorical (Ward).

    - fit(X, y): computes smoothed mean(y) per category using ONLY the rows passed in.
    - transform(X): maps categories to their fitted encoding. Categories not seen during
      fit (e.g. a ward that only appears in a held-out fold) fall back to the global mean.
    - smoothing: pulls low-count wards toward the global mean (Bayesian-style shrinkage),
      so a ward with only 3 historical records doesn't get an extreme 0.0 or 1.0 encoding.
    """
    def __init__(self, smoothing=10.0):
        self.smoothing = smoothing

    def fit(self, X, y):
        ward = pd.Series(np.asarray(X).ravel(), name='ward')
        target = pd.Series(np.asarray(y).ravel(), name='target')
        frame = pd.concat([ward, target], axis=1)

        self.global_mean_ = float(target.mean())
        agg = frame.groupby('ward')['target'].agg(['mean', 'count'])
        smoothed = (agg['mean'] * agg['count'] + self.global_mean_ * self.smoothing) \
                   / (agg['count'] + self.smoothing)
        self.mapping_ = smoothed.to_dict()
        return self

    def transform(self, X):
        ward = pd.Series(np.asarray(X).ravel(), name='ward')
        return ward.map(self.mapping_).fillna(self.global_mean_).to_numpy().reshape(-1, 1)


## 3. Preprocessing pipeline (leakage-safe)

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('ward_te', Pipeline([
        ('encode', WardTargetEncoder(smoothing=10.0)),
        ('scale', StandardScaler())
    ]), ward_col),
    ('num_scale', StandardScaler(), numeric_cols),
    ('rest', 'passthrough', passthrough_cols),
])

# Feature names in the ORDER the ColumnTransformer will output them --
# needed later for SHAP and the thresholding loop.
transformed_feature_names = ['WARD_TE'] + numeric_cols + passthrough_cols
print(transformed_feature_names)


## 4. Outer train/test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('Train:', X_train.shape, ' Test:', X_test.shape)


## 5. Model definition + GridSearchCV

**This is the block to swap out per model.** Replace `DecisionTreeClassifier` and `param_grid` with the classifier/grid from your RF / XGBoost / KNN / LR / MLP / GaussianNB notebooks. Leave the `Pipeline(...)` wrapper and `GridSearchCV` call as-is -- that's what makes the ward encoding refit correctly inside every CV fold.

In [ ]:
clf = DecisionTreeClassifier(random_state=42)

param_grid = {
    'clf__max_depth': [None, 5, 10, 15, 20],
    'clf__min_samples_split': [2, 5, 10],
    'clf__min_samples_leaf': [1, 2, 4],
    'clf__criterion': ['gini', 'entropy'],
}

full_pipeline = Pipeline([
    ('prep', preprocessor),
    ('clf', clf),
])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=full_pipeline,
    param_grid=param_grid,
    cv=skf,
    scoring='f1',
    n_jobs=-1,
)
grid_search.fit(X_train, y_train)

print('Best params:', grid_search.best_params_)
print('Best CV F1 :', grid_search.best_score_)

best_pipeline = grid_search.best_estimator_


## 6. Evaluate baseline on the FULL test set

In [ ]:
y_pred = best_pipeline.predict(X_test)
y_proba = best_pipeline.predict_proba(X_test)[:, 1]

print('Accuracy :', round(accuracy_score(y_test, y_pred) * 100, 2))
print('ROC-AUC  :', round(roc_auc_score(y_test, y_proba), 4))
print('PR-AUC   :', round(average_precision_score(y_test, y_proba), 4))
print()
print(classification_report(y_test, y_pred, target_names=['Dengue (0)', 'Malaria (1)']))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Dengue (0)', 'Malaria (1)'],
            yticklabels=['Dengue (0)', 'Malaria (1)'])
plt.title('Confusion Matrix -- Decision Tree (leakage-safe, full test set)')
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.tight_layout()
plt.savefig('dt_cm_leakage_safe.png', dpi=400)
plt.show()


## 7. Transform train/test into the model's actual numeric feature space

This gives us a `WARD_TE` column (leakage-safe) instead of the old leaked `WARD_encoded`, with the SAME semantics your original code used downstream for SHAP and the thresholding loop.

In [ ]:
X_train_t = pd.DataFrame(
    best_pipeline.named_steps['prep'].transform(X_train),
    columns=transformed_feature_names, index=X_train.index
)
X_test_t = pd.DataFrame(
    best_pipeline.named_steps['prep'].transform(X_test),
    columns=transformed_feature_names, index=X_test.index
)

best_model = best_pipeline.named_steps['clf']   # the fitted classifier only, for SHAP
X_train_t.head()


## 8. SHAP global attributions

In [ ]:
import shap

explainer = shap.TreeExplainer(best_model)
shap_values = explainer(X_test_t)          # full test set, not a 50-row subset
shap_values_positive = shap_values[..., 1]

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_positive, X_test_t,
                   feature_names=transformed_feature_names, show=False)
plt.title('SHAP Beeswarm -- Positive Class (Malaria), leakage-safe features')
plt.tight_layout()
plt.savefig('shap_beeswarm_dt_leakage_safe.png', dpi=400, bbox_inches='tight')
plt.show()

shap_vals_all = shap_values.values
mean_abs_shap_class_0 = np.mean(np.abs(shap_vals_all[:, :, 0]), axis=0)
mean_abs_shap_class_1 = np.mean(np.abs(shap_vals_all[:, :, 1]), axis=0)
mean_abs_shap_global = (mean_abs_shap_class_0 + mean_abs_shap_class_1) / 2

shap_importance_df = pd.DataFrame({
    'Feature': transformed_feature_names,
    'Mean_Abs_SHAP': mean_abs_shap_global
}).sort_values('Mean_Abs_SHAP', ascending=False)

shap_importance_df


## 9. Tau-thresholding loop -- FULL TEST SET evaluation

This is the corrected version of your RQ2 sweep. Same logic as before (drop features with mean |SHAP| <= tau, retrain, measure performance), but every threshold is now evaluated on the entire held-out test set.

In [ ]:
shap_thresholds = sorted(np.unique(mean_abs_shap_global))

final_results_shap_dt = []

for t in shap_thresholds:
    mask = mean_abs_shap_global >= float(t)
    selected_features = [f for f, m in zip(transformed_feature_names, mask) if m]
    if len(selected_features) == 0:
        continue

    X_train_sel = X_train_t[selected_features]
    X_test_sel  = X_test_t[selected_features]

    skf_inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    dt_inner = DecisionTreeClassifier(random_state=42)
    grid_inner = GridSearchCV(
        estimator=dt_inner,
        param_grid={
            'max_depth': [None, 5, 10, 15, 20],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4],
            'criterion': ['gini', 'entropy'],
        },
        cv=skf_inner, scoring='f1', n_jobs=-1,
    )
    grid_inner.fit(X_train_sel, y_train)
    model_t = grid_inner.best_estimator_

    # --- FULL TEST SET, not .iloc[:10] / .iloc[:50] ---
    y_pred_t = model_t.predict(X_test_sel)
    y_proba_t = model_t.predict_proba(X_test_sel)[:, 1]

    acc = accuracy_score(y_test, y_pred_t)
    f1_macro = f1_score(y_test, y_pred_t, average='macro')
    f1_dengue = f1_score(y_test, y_pred_t, pos_label=0)
    roc_auc = roc_auc_score(y_test, y_proba_t)
    pr_auc = average_precision_score(y_test, y_proba_t)

    final_results_shap_dt.append({
        'Threshold': float(t),
        'Num_Features': len(selected_features),
        'Selected_Features': selected_features,
        'Accuracy': round(acc * 100, 2),
        'Macro_F1': round(f1_macro, 4),
        'Dengue_F1': round(f1_dengue, 4),
        'ROC_AUC': round(roc_auc, 4),
        'PR_AUC': round(pr_auc, 4),
    })

    print(f"tau={t:.4f} | features={len(selected_features)} | "
          f"MacroF1={f1_macro:.4f} | DengueF1={f1_dengue:.4f} | PR-AUC={pr_auc:.4f}")

results_df = pd.DataFrame(final_results_shap_dt)
results_df


## 10. (Optional but recommended) 5-fold CV robustness check

Reports mean ± std across 5 folds for your headline baseline numbers, so the paper isn't relying on a single 80/20 split's point estimate.

In [ ]:
cv_pipeline = Pipeline([
    ('prep', preprocessor),
    ('clf', DecisionTreeClassifier(**{k.replace('clf__', ''): v
                                       for k, v in grid_search.best_params_.items()},
                                    random_state=42)),
])

scoring = {
    'accuracy': 'accuracy',
    'f1_macro': 'f1_macro',
    'roc_auc': 'roc_auc',
    'average_precision': 'average_precision',
}

cv_scores = cross_validate(cv_pipeline, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=42),
                            scoring=scoring, n_jobs=-1)

for metric in scoring:
    vals = cv_scores[f'test_{metric}']
    print(f'{metric:20s}: {vals.mean():.4f} +/- {vals.std():.4f}')
